Астромтерическая калибровка снимка с помощью линейной модели

In [1]:
from math import *
import numpy as np

import pandas as pd
import requests
import io

from astropy.io import fits
from astropy.wcs import WCS
from astropy.time import Time
from astropy.time import TimeDelta

from scipy.optimize import curve_fit

from astropy import units as u
from astropy.coordinates import SkyCoord

import glob
import os
import re

import matplotlib.pyplot as plt
import matplotlib
from matplotlib import cm
rgray = matplotlib.colormaps['gray'].reversed()
%matplotlib widget

In [2]:
# сортированный список файлов
# цикл, в котором перебираются все эти файлы позволяет вести обработку сразу большого числа снимков
upath = '../game1/fits/'# это путь к папке с файлами (можно потом заменить на нужный)
filelist = sorted(glob.glob(f'{upath}Uranus*wcs.fits'))
print('{}'.format(filelist))

['../game1/fits/Uranus_R_4_0001_wcs.fits', '../game1/fits/Uranus_R_4_0002_wcs.fits', '../game1/fits/Uranus_R_4_0003_wcs.fits', '../game1/fits/Uranus_R_4_0004_wcs.fits', '../game1/fits/Uranus_R_4_0005_wcs.fits']


In [9]:
# Ниже будет пример работы с одним из файлов в данном списке
filename = '../game1/fits/Uranus_R_4_0002_wcs.fits'
hdul = fits.open(filename)
img = (hdul[0].data).astype(float) # считываем кадр в переменную img
h, w = np.shape(img)# размеры изображения в пикселях по вертикали и горизонтали

In [10]:
# нарисуем картинку и пометим на ней звезды, Уран и его спутники
ap =10 # размер апертуры (кружок вокруг звезды)
cf1,cf2 = 0.1,0.5 # определяет окно значений отсчетов, которые будут отрисованы (смотри plt.imshow дальше в этом блоке) 

fig, ax = plt.subplots(figsize=(3.0,3.0), dpi=300)
ax.xaxis.set_tick_params(labelsize=5)
ax.yaxis.set_tick_params(labelsize=5)
plt.xlabel('X, pix', fontsize = 4)
plt.ylabel('Y, pix', fontsize=4)
#рисуем изображение в серой шкале. Можно поменять cmap='gray' на что-нибудь еще и можно получить всякие более модные отображения
plt.imshow(img, vmin=np.median(img)- cf1*np.std(img) ,
           vmax=np.median(img)  + cf2*np.std(img), origin='lower', cmap=rgray) 
#автоматически подгоняем рамку 
plt.tight_layout()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [11]:
t = Time(hdul[0].header['DATE-OBS'], format='isot') # момент начала съемки
t+=TimeDelta(float(hdul[0].header['EXPTIME']), format='sec')/2 # добавление к начальному моменту половины времени накопления
# То есть t теперь - это центральный момент экспозиции. Потом для него надо будет считать эфемериды 
JD = float(t.copy(format='jd').value) # Юлианская дата для центрального момента
yr = float(t.copy(format='jyear').value) # Юлианский год для центрального момента (нужен для учета собственных движений звезд)

RAc,DECc = hdul[0].header['CRVAL1'],hdul[0].header['CRVAL2'] # экваториальные координаты геометрического центра кадра
# потом эти RAc,DECc будут нужны и для выборки из Gaia, и для расчета тангенциальных координат при калибровке снимка
limmag1,limmag2 = 5.0,16.5 # диапазон блеска для опорных звезд
fov = 0.4# это размер области в градусах

In [12]:
# выборка данных из Gaia DR3 http://sfa.puldb.ru:9810
response = requests.get(f'http://192.168.30.9:8100?cmd=box&ra={RAc}&dec={DECc}&fov={fov}')
# response = requests.get(f'http://sfa.puldb.ru:9810?cmd=box&ra={RAc}&dec={DECc}&fov={fov}')
raw_data = response.content
df = pd.read_csv(io.StringIO(raw_data.decode('utf-8')))
# В последнее время сбои при выборки с родных серверов Gaia стали частыми. 
# Поэтому тут реализована выборка из нашей локальной версии Gaia, размещенной на нашем сервере
# df - это табличка (датафрейм в формате pandas)
df

,solution_id,designation,source_id,random_index,ref_epoch,ra,ra_error,dec,dec_error,parallax,...,azero_gspphot,azero_gspphot_lower,azero_gspphot_upper,ag_gspphot,ag_gspphot_lower,ag_gspphot_upper,ebpminrp_gspphot,ebpminrp_gspphot_lower,ebpminrp_gspphot_upper,libname_gspphot
0,1636148068921376768,Gaia DR3 51258373093921280,51258373093921280,345028988,2016.0,59.125298,0.216448,19.957065,0.137909,0.021883,...,0.0148,0.0033,0.0432,0.0112,0.0025,0.0326,0.0060,0.0013,0.0176,MARCS
1,1636148068921376768,Gaia DR3 51258510532872192,51258510532872192,400992464,2016.0,59.082375,0.369506,19.953836,0.234851,1.017526,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1636148068921376768,Gaia DR3 51258544892612096,51258544892612096,1382546787,2016.0,59.106924,0.133514,19.955876,0.104925,1.340692,...,0.5511,0.5178,0.5837,0.3811,0.3580,0.4038,0.2094,0.1969,0.2216,PHOENIX
3,1636148068921376768,Gaia DR3 51258549189666176,51258549189666176,1229386944,2016.0,59.110723,0.106526,19.959347,0.075979,0.357447,...,0.0329,0.0071,0.0939,0.0253,0.0054,0.0726,0.0135,0.0029,0.0389,PHOENIX
4,1636148068921376768,Gaia DR3 51258583549403008,51258583549403008,1224435394,2016.0,59.114679,0.040356,19.963272,0.031519,0.425562,...,0.8491,0.8048,0.8709,0.6908,0.6540,0.7091,0.3726,0.3528,0.3824,PHOENIX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1187,1636148068921376768,Gaia DR3 51353107189730944,51353107189730944,1439971659,2016.0,58.839558,0.052091,20.350220,0.033324,0.507913,...,0.0653,0.0239,0.2034,0.0517,0.0189,0.1621,0.0279,0.0102,0.0871,PHOENIX
1188,1636148068921376768,Gaia DR3 51353240331884416,51353240331884416,840026978,2016.0,58.883346,0.254405,20.350842,0.175563,0.439484,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1189,1636148068921376768,Gaia DR3 51353244627240960,51353244627240960,1023939329,2016.0,58.882036,0.894397,20.352231,0.680267,0.744170,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1190,1636148068921376768,Gaia DR3 51353790087696640,51353790087696640,322811323,2016.0,58.977139,0.744305,20.347207,0.516746,-0.337609,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# стандартный набор - функции для расчета углов, тангенциальных координат (и обратное преобразование)
def angular_distance(ra,dec,RA,DEC):
    return acos(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))

def tangential_coordinates(ra,dec,RA,DEC):
    ksi = cos(dec)*sin(ra-RA)/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    eta = (sin(dec)*cos(DEC)-cos(dec)*sin(DEC)*cos(ra-RA))/(sin(dec)*sin(DEC)+cos(dec)*cos(DEC)*cos(ra-RA))
    return ksi,eta

def equatorial_coordinates(ksi,eta,RA,Dec):
    x,y,z = np.dot(np.array([[-sin(RA),-cos(RA) * sin(Dec),cos(RA) * cos(Dec)],
                             [cos(RA),-sin(RA) * sin(Dec),sin(RA) * cos(Dec)],
                             [0,cos(Dec),sin(Dec)]]),
                   np.array([ksi,eta,1]))/sqrt(1+ksi*ksi+eta*eta)
    ra = atan2(y,x)
    dec = atan2(z,sqrt(x*x+y*y))
    if(ra<0):
        ra+=2*pi
    return ra,dec
# функция, которая работает с таблицей, полученной в результате выборки из Gaia
# производится учет собственных движений и возвращаются три массива (два для координат и один для оценок блеска звезд)
def get_Gaia_data(yr, lm1,lm2, gdf):
    ra,dec,gmag = [],[],[]
    for index, row in gdf.iterrows():
        G = row['phot_g_mean_mag']
        if (not isnan(G)) and (lm1<G<lm2):
            ra0,dec0,pmra,pmdec,ep = radians(row['ra']),radians(row['dec']),\
                radians(row['pmra']/3600000.0),\
                radians(row['pmdec']/3600000.0),\
                row['ref_epoch']
            if not isnan(pmra):
                alpha,delta = np.degrees(equatorial_coordinates(pmra*(yr-ep),pmdec*(yr-ep),ra0,dec0))
                ra.append(alpha)
                dec.append(delta)
                gmag.append(G)
    return np.array(ra),np.array(dec),np.array(gmag)

In [14]:
ra,dec,gmag = get_Gaia_data(yr,limmag1,limmag2, df) # после вызова этой функции у нас в руках экваториальные координаты с учетом собственных движений 

In [15]:
W = WCS(hdul[0].header)#заголовок WCS

Set MJD-END to 60939.945681 from DATE-END'. [astropy.wcs.wcs]


In [16]:
bdr = 100 # размер поребрика (бордюра) в пикселях (чтобы не отбирать звезды слишком близкие к краю кадра)
# считаем пиксельные координаты отобранных звезд Gaia DR3 с помощью WCS. 
pxpos = W.wcs_world2pix(np.stack((ra, dec), axis=-1),1)
# p_indx - это индексы звезд, попавших в кадр
p_indx = np.where((pxpos[:,0]>bdr) & (pxpos[:,1]>bdr) & (pxpos[:,0]<w-bdr) & (pxpos[:,1]<h-bdr))
# отбираем по индексам только звезды, попавшие в кадр
pixpos = pxpos[p_indx]
# отбираем в массивы экваториальных координат и блеска только данные для звезд в пределах кадра (с индексами p_indx)
ra,dec,gmag = ra[p_indx],dec[p_indx],gmag[p_indx]

In [18]:
# нарисуем картинку и пометим на ней звезды, Уран и его спутники
ap =10 # размер апертуры (кружок вокруг звезды)
cf1,cf2 = 0.1,0.5 # определяет окно значений отсчетов, которые будут отрисованы (смотри plt.imshow дальше в этом блоке) 
wcs_based = True# определяет как рисовать систему координат. 
#True - использует WCS и отображаются экваториальные координаты 
#False - отображаются пиксельные координаты 

if wcs_based:
    fig, ax = plt.subplots(figsize=(3.0,3.0), dpi=300, tight_layout=True, subplot_kw={'projection': W})
    ax.coords[0].set_ticklabel(size=6)
    ax.coords[1].set_ticklabel(size=6)
    plt.xlabel('RA', fontsize = 5)
    plt.ylabel('DEC', fontsize=5)
else:
    fig, ax = plt.subplots(figsize=(3.0,3.0), dpi=300)
    ax.xaxis.set_tick_params(labelsize=5)
    ax.yaxis.set_tick_params(labelsize=5)
    plt.xlabel('X, pix', fontsize = 4)
    plt.ylabel('Y, pix', fontsize=4)
#рисуем изображение в серой шкале. Можно поменять cmap='gray' на что-нибудь еще и можно получить всякие более модные отображения
plt.imshow(img, vmin=np.median(img)- cf1*np.std(img) ,
           vmax=np.median(img)  + cf2*np.std(img), origin='lower', cmap=rgray) 
#рисуем метки звезд согласно WCS и координатам звезд из Gaia EDR3
plt.scatter(pixpos[:,0], pixpos[:,1], s=ap, facecolors='none', edgecolors='black', linewidth=0.2)
#автоматически подгоняем рамку 
plt.tight_layout()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [20]:
# функции для фиттинга изображений звезд и вычисления их пиксельных координат
def gaussian_profile(box,*p):
    Ibkg,Imax,A,B,C,xc,yc = p
    x = np.arange(0, box, 1)
    y = np.arange(0, box, 1)
    x, y = np.meshgrid(x, y)
    r = A*(x-xc)**2+C*(y-yc)**2+B*(x-xc)*(y-yc)
    return np.ravel(Ibkg+Imax*np.exp(-r))

def im_centroiding(I):
    x_c = int(np.shape(I)[0]/2)
    y_c = x_c
    Ib, Im = np.median(I),np.max(I) 
    A = C = 1/5
    B = 0
    return curve_fit(f=gaussian_profile, p0=np.array([Ib,Im,A,B,C,x_c,y_c]),\
                   xdata = np.shape(I)[0],ydata=I.ravel(),maxfev=1000)

In [40]:
ap = 10
xp,yp = np.array([]),np.array([])
ksi,eta = np.array([]),np.array([])
for k in range(np.size(gmag)):
    x,y = int(pixpos[k,0]),int(pixpos[k,1])
    I = img[y-ap:y+ap,x-ap:x+ap]
    try:
        p, covp = im_centroiding(I)
    except:
        continue
   Ibkg,Imax,A,B,C,xc,yc =  p
    if sqrt(covp[5,5])<0.25 and sqrt(covp[6,6])<0.25:
        xp,yp = np.append(xp,x-ap+xc),np.append(yp,y-ap+yc)
        # добавить свою функцию для вычисления тангенциальных координат
        tx,ty = 0,0
        ksi,eta = np.append(ksi,tx),np.append(eta,ty)

Задача: произвести астрометрическую калибровку (астрометрическую редукцию) кадра с помощью линейной модели 